## Deployment — resposta à pergunta de negócio

A célula abaixo gera uma conclusão automática com os números obtidos neste arquivo. Em uma aplicação industrial, os rótulos finais dos regimes devem ser revisados com especialistas de processo e, se existirem, dados de carga, equipamento ligado/desligado, produção, manutenção e falhas conhecidas.

In [ ]:
summary = []
for row in profile_rows:
    summary.append((row['Regime'], row['%'], row.get('CO(GT)', np.nan), row.get('C6H6(GT)', np.nan), row.get('NOx(GT)', np.nan)))

print('RESPOSTA EXECUTIVA')
print('-' * 80)
print(f'1) Os dados suportam {best_k} macro-regime(s) como solução mais separável.')
for r, share, co, benz, nox in summary:
    print(f'   • Regime {r}: {share:.1f}% das horas | CO mediano={co:.2f} | C6H6 mediano={benz:.2f} | NOx mediano={nox:.1f}')
print(f'2) A variável descartada por ausência excessiva foi: {", ".join(dropped_features)}.')
print(f'3) Melhor tratamento de ausências no teste: {best_imputer_name}.')
print(f'4) Foram identificados {anomaly_flag.sum()} candidatos a anomalia ({100*anomaly_flag.mean():.2f}%), já descontando linhas de baixa qualidade.')
print(f'5) Há {low_quality.sum()} linhas que devem ser tratadas como problema de qualidade, não como falha/anomalia.')
print('\nInterpretação:')
print('• Uma mudança de Regime 1 para Regime 2, por si só, NÃO é uma anomalia: é uma mudança para outro estado recorrente.')
print('• O alerta mais confiável é o desvio extremo DENTRO do regime atual.')
print('• Como este conjunto é de qualidade do ar/sensores, “anomalia” significa comportamento estatisticamente atípico; confirmar falha física exige cruzamento com eventos operacionais/manutenção.')

### Como levar para produção

Um fluxo operacional recomendado seria:

`novos sensores → qualidade de dados → imputação → RobustScaler → classificação do regime → detector específico do regime → persistência do alerta → operador/manutenção`

**Regras práticas sugeridas:**
- não alertar quando a qualidade de dados estiver abaixo do limite;
- apresentar o regime atual e a probabilidade/score de anomalia separadamente;
- exigir persistência, por exemplo **2 de 3 amostras consecutivas**, antes de abrir um alerta de manutenção;
- registrar transições e anomalias para posterior comparação com ordens de manutenção e falhas reais;
- recalibrar clusters e limiares periodicamente se o processo sofrer mudanças estruturais.

### Limitação importante
O notebook descobre padrões estatísticos nos dados disponíveis. Ele não consegue afirmar sozinho que um ponto extremo correspondeu a uma **falha de equipamento**. Para transformar “candidato estatístico” em “falha confirmada”, é necessário cruzar as datas com histórico de alarmes, eventos, ordens de manutenção ou registro de falhas.